## How much does the tolerance impact the bias in the estimate of the posterior standard deviation?
In this notebook, we explore running stan with different convergence tolerances on our conjugate inverse-gamma / Gaussian (known mean) example test case to explore what the impact of converging early is on the bias of the variational approximation of the posterior standard deviation relative to the best variational approximation.

We're going to run stan's VI with default (1) MC samples and adjusted (100) MC samples at convergence tolerances of $10^{-6}$, $10^{-5}$, $10^{-4}$, $10^{-3}$, $10^{-2}$ (stan's default value), $10^{-1}$, and $1$.

In [1]:
# imports and model setup
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/invgamma_gaussianknownmean/random_restarts")
STAN_FILE_NAME = "stan_invgamma_gaussian_knownmean.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")

Notebook dir: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/invgamma_gaussianknownmean/random_restarts
Stan model: /Users/madelynandersen/Documents/VB Bakeoff/simpleVI/single_MC/invgamma_gaussianknownmean/random_restarts/stan_invgamma_gaussian_knownmean.stan
Python executable: /Users/madelynandersen/miniconda3/envs/stan3/bin/python
CmdStan path: /Users/madelynandersen/miniconda3/envs/stan3/bin/cmdstan


/Users/madelynandersen/miniconda3/envs/stan3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modulars import gaussian_1d
from modulars.utils import load_config, load_best_values, exp_lognormal_moments

config_file = NOTEBOOK_DIR.parent / "invgamma_config.json"
config = load_config(config_file)

mu_like = config["mu_like"]
sigma_like = config["sigma_like"]
alpha_prior = config["alpha_prior"]
beta_prior = config["beta_prior"]
n_samples = int(config["n_samples"])

data = gaussian_1d(n_samples, mu_like, sigma_like=sigma_like).astype(float)

stan_data = {
    "N": len(data),
    "y": data,
    "mu_like": mu_like,
    "alpha_prior": alpha_prior,
    "beta_prior": beta_prior,
}
PARAM_COLUMNS = ["sigma_sq"]
LOG_COLUMNS = []
best_mu, best_std = load_best_values(config=config, transform=exp_lognormal_moments, n_samples=50_000, seed=1)
best_mu, best_std

(0.5308105941122779, 0.181962351468537)

In [3]:
# set up the multiple runs and run the model
import cmdstanpy


tols = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
default_post_var_approx_outputs = {} # save as tolerance: [post mean, post std]
adjusted_post_var_approx_outputs = {} # save as tolerance: [post mean, post std]

for tol in tols:
    stan_model = cmdstanpy.CmdStanModel(stan_file=STAN_FILE)
    default_fit = stan_model.variational(
    data=stan_data,
    tol_rel_obj=tol,
    seed=3,
    draws=50_000,
    require_converged=False,
    iter=100_000
    )
    idx = default_fit.column_names.index('sigma_sq')
    default_post_var_approx_outputs[tol] = [default_fit.variational_sample[:, idx].mean(), default_fit.variational_sample[:, idx].std()]

    adjusted_fit = stan_model.variational(
    data=stan_data,
    tol_rel_obj=tol,
    seed=3,
    draws=50_000,
    require_converged=False,
    grad_samples = 100,
    iter=100_000
    )
    idx = adjusted_fit.column_names.index('sigma_sq')
    adjusted_post_var_approx_outputs[tol] = [adjusted_fit.variational_sample[:, idx].mean(), adjusted_fit.variational_sample[:, idx].std()]


16:33:22 - cmdstanpy - INFO - Chain [1] start processing


16:33:22 - cmdstanpy - INFO - Chain [1] done processing


16:33:22 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


16:33:22 - cmdstanpy - INFO - Chain [1] start processing


16:33:24 - cmdstanpy - INFO - Chain [1] done processing


...

16:33:33 - cmdstanpy - INFO - Chain [1] done processing


16:33:33 - cmdstanpy - INFO - Chain [1] start processing


16:33:33 - cmdstanpy - INFO - Chain [1] done processing


16:33:33 - cmdstanpy - INFO - Chain [1] start processing


16:33:34 - cmdstanpy - INFO - Chain [1] done processing


3

In [4]:
    # post_var_approx_outputs[tol] = [fit.stan_variable("theta").mean(), fit.stan_variable("theta").std()]

## Bias in the posterior standard deviation vs. tolerance

We plot the percent error of each run's variational approximation of the posterior standard deviation relative to the best variational approximation (`best_std`), as a function of the convergence tolerance used.

In [ ]:
import plotly.graph_objects as go

sorted_tols = sorted(tols)

default_pct_error = [
    (default_post_var_approx_outputs[tol][1] - best_std) / best_std * 100
    for tol in sorted_tols
]
adjusted_pct_error = [
    (adjusted_post_var_approx_outputs[tol][1] - best_std) / best_std * 100
    for tol in sorted_tols
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sorted_tols, y=default_pct_error, mode='lines+markers',
    line=dict(color='blue'), marker=dict(color='blue'),
    name='1 MC sample (default)',
))
fig.add_trace(go.Scatter(
    x=sorted_tols, y=adjusted_pct_error, mode='lines+markers',
    line=dict(color='purple'), marker=dict(color='purple'),
    name='100 MC samples (adjusted)',
))
fig.add_vline(x=0.01, line=dict(color='black', dash='dash'))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines', line=dict(color='black', dash='dash'),
    name='default tolerance',
))

fig.update_xaxes(
    title_text='Convergence tolerance (tol_rel_obj)', type='log',
    tickmode='array', tickvals=sorted_tols,
    ticktext=[f'{t:.0e}' for t in sorted_tols],
)
fig.update_yaxes(title_text='Percent error in posterior std vs. best approximation (%)')
fig.update_layout(
    title='Bias in the variational approximation of the posterior std vs. tolerance',
    height=600, width=950,
)
fig.show()